<a href="https://colab.research.google.com/github/samoliverramos/FIAP2026/blob/main/aulas/Aula_Perceptron_Introducao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Redes Neurais Artificiais – Aula: Perceptron na Prática

Copuyright Prof. PhD in progress Samir Ramos

## Objetivos de aprendizado
- Entender o que é um neurônio artificial (Perceptron).
- Implementar o feedforward de um perceptron.
- Treinar um perceptron ajustando pesos com base nos erros.
- Visualizar a fronteira de decisão antes e depois do treino.
- Explorar efeitos de taxa de aprendizado e número de épocas.

---

Você é um analista de dados em uma **loja online**.  
Com base em duas informações simples sobre o comportamento do usuário:

- `tempo_no_site` (minutos)
- `valor_medio_carrinho` (R$)

queremos classificar se ele **tende a comprar (1)** ou **não comprar (0)**.

Hoje vamos treinar um **único neurônio (Perceptron)** para fazer essa classificação, visualizando tudo em 2D.

2. Setup: imports e checagem de ambiente

In [ ]:
# Imports básicos
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Configurações de plot (opcional)
plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True

3. Gerar (ou carregar) dados sintéticos 2D
Aqui vamos gerar dados simples e bem visualizáveis.

In [ ]:
# Vamos gerar um dataset sintético 2D para classificação binária

np.random.seed(42)  # para resultados reproduzíveis

# Classe 0: não compra
tempo_0 = np.random.normal(loc=5, scale=1.0, size=50)   # minutos
valor_0 = np.random.normal(loc=50, scale=10.0, size=50) # R$
y0 = np.zeros(50)

# Classe 1: compra
tempo_1 = np.random.normal(loc=10, scale=1.0, size=50)
valor_1 = np.random.normal(loc=150, scale=10.0, size=50)
y1 = np.ones(50)

# Empilhar tudo
X = np.column_stack([
    np.concatenate([tempo_0, tempo_1]),
    np.concatenate([valor_0, valor_1])
])
y = np.concatenate([y0, y1])

print("Formato de X:", X.shape)
print("Formato de y:", y.shape)
print("Primeiros 5 elementos de X:\n", X[:5])
print("Primeiros 5 elementos de y:\n", y[:20])

4. Visualizar os dados (EDA mínima)

In [ ]:
# Scatter plot: cada ponto é um "cliente"
plt.figure()
plt.scatter(X[y == 0, 0], X[y == 0, 1], color="red", label="Não comprou (0)")
plt.scatter(X[y == 1, 0], X[y == 1, 1], color="blue", label="Comprou (1)")
plt.xlabel("Tempo no site (min)")
plt.ylabel("Valor médio do carrinho (R$)")
plt.legend()
plt.title("Clientes - Classes 0 e 1")
plt.show()

**Comentário didático**

- Cada ponto é um cliente.
- O eixo x é o **tempo no site**.
- O eixo y é o **valor médio do carrinho**.
- Em vermelho, clientes que **não compraram (0)**.
- Em azul, clientes que **compraram (1)**.

A ideia do Perceptron: encontrar **uma linha** que separe bem os vermelhos dos azuis.
Essa linha será a nossa **fronteira de decisão**.

5. Micro-teoria: definindo o neurônio (feedforward)

## Perceptron: um neurônio artificial

Um neurônio simples recebe entradas e produz uma saída:

- Entradas: \|$x_1, x_2\$| (no nosso caso: tempo e valor do carrinho)
- Pesos: \|$w_1, w_2\$| (o quanto cada entrada "pesa" na decisão)
- Bias: \|$b\$| (um ajuste global)

Cálculo:

\|$
z = w_1 x_1 + w_2 x_2 + b
\$|

Depois aplicamos uma **função de ativação**:

\|$
\hat{y} =
\begin{cases}
1, & \text{se } z \ge 0 \\
0, & \text{se } z < 0
\end{cases}
\$|

Hoje vamos usar essa função degrau (step), típica do Perceptron clássico.

6. Implementar feedforward do Perceptron

In [ ]:
def step_function(z):
    """
    Função de ativação degrau.
    Retorna 1 se z >= 0, senão 0.
    """
    return np.where(z >= 0, 1, 0)

def perceptron_predict(X, weights, bias):
    """
    X: array de shape (n_amostras, n_features)
    weights: array de shape (n_features,)
    bias: escalar

    Retorna: predições 0 ou 1
    """
    z = np.dot(X, weights) + bias
    y_hat = step_function(z)
    return y_hat

**Explicação rápida**

- `np.dot(X, weights)` faz a soma ponderada das entradas.
- `+ bias` é o ajuste global.
- `step_function` decide se o neurônio "dispara" (1) ou não (0).

7. Visualizar a fronteira de decisão (função auxiliar)

In [ ]:
def plot_decision_boundary(X, y, weights, bias, title="Fronteira de decisão"):
    """
    Plota os dados e a linha de decisão do perceptron em 2D.
    """
    # Plot dos pontos
    plt.figure()
    plt.scatter(X[y == 0, 0], X[y == 0, 1], color="red", label="Não comprou (0)")
    plt.scatter(X[y == 1, 0], X[y == 1, 1], color="blue", label="Comprou (1)")

    # Cria uma grade de valores para x1
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x1_vals = np.linspace(x1_min, x1_max, 100)

    # Se w2 != 0, calculamos x2 pela equação da fronteira:
    # w1*x1 + w2*x2 + b = 0  =>  x2 = -(w1*x1 + b)/w2
    if weights[1] != 0:
        x2_vals = -(weights[0] * x1_vals + bias) / weights[1]
        plt.plot(x1_vals, x2_vals, color="green", label="Fronteira de decisão")

    plt.xlabel("Tempo no site (min)")
    plt.ylabel("Valor médio do carrinho (R$)")
    plt.legend()
    plt.title(title)
    plt.show()

8. Inicializar pesos aleatórios e ver a linha inicial

In [ ]:
# Inicializando pesos e bias aleatórios
np.random.seed(42)
weights = np.random.randn(2)  # para 2 features
bias = np.random.randn()

print("Pesos iniciais:", weights)
print("Bias inicial:", bias)

# Plotar a fronteira de decisão inicial (antes do treino)
plot_decision_boundary(X, y, weights, bias, title="Antes do treino")

Observe que, com pesos aleatórios, a linha provavelmente **não separa bem** as classes.
O papel do treino é **ajustar esses pesos** para melhorar a separação.

9. Treino do Perceptron (regra de atualização simples)

## Treinando o Perceptron

Ideia intuitiva:

1. Fazemos a predição para um exemplo.
2. Comparamos com o rótulo verdadeiro.
3. Se errou, ajustamos os pesos na direção do erro.

Regra de atualização (para cada exemplo):

\|$
w_j \leftarrow w_j + \eta \cdot (y - \hat{y}) \cdot x_j
\$|

\|$
b \leftarrow b + \eta \cdot (y - \hat{y})
\$|

onde:
- \|$\eta\$| é a **taxa de aprendizado** (learning rate)
- \|$y\$| é o rótulo verdadeiro
- \|$\hat{y}\$| é a predição do perceptron

In [ ]:
def train_perceptron(X, y, lr=0.01, n_epochs=25):
    """
    Treina um perceptron usando a regra clássica de atualização.

    X: (n_amostras, n_features)
    y: (n_amostras,)
    lr: taxa de aprendizado (learning rate)
    n_epochs: número de vezes que percorremos todo o dataset

    Retorna: weights, bias, history
    """
    n_samples, n_features = X.shape

    # Inicialização aleatória
    np.random.seed(42)
    weights = np.random.randn(n_features)
    bias = np.random.randn()

    history = {
        "epoch": [],
        "errors": []
    }

    for epoch in range(n_epochs):
        errors = 0

        for xi, yi in zip(X, y):
            # Predição para uma única amostra
            y_hat = perceptron_predict(xi.reshape(1, -1), weights, bias)[0]

            # Erro
            update = lr * (yi - y_hat)

            if update != 0:
                # Ajuste de pesos e bias
                weights += update * xi
                bias += update
                errors += 1

        history["epoch"].append(epoch)
        history["errors"].append(errors)

        # Opcional: printar progresso
        print(f"Época {epoch+1}/{n_epochs} - Erros de classificação: {errors}")

    return weights, bias, history

10. Rodar o treinamento e visualizar resultados

In [ ]:
learning_rate = 0.01
n_epochs = 25

weights_trained, bias_trained, history = train_perceptron(
    X, y, lr=learning_rate, n_epochs=n_epochs
)

print("\nPesos treinados:", weights_trained)
print("Bias treinado:", bias_trained)

In [ ]:
# Plot da evolução dos erros por época
plt.figure()
plt.plot(history["epoch"], history["errors"], marker="o")
plt.xlabel("Época")
plt.ylabel("Erros de classificação")
plt.title("Evolução dos erros ao longo do treinamento")
plt.show()

In [ ]:
# Fronteira de decisão depois do treino
plot_decision_boundary(X, y, weights_trained, bias_trained, title="Depois do treino")

11. Avaliar acurácia final

In [ ]:
# Avaliando performance
y_pred = perceptron_predict(X, weights_trained, bias_trained)
accuracy = (y_pred == y).mean()
print(f"Acurácia no conjunto completo: {accuracy*100:.2f}%")

################################################################

ANTES DE INCIARMOS ALGUNS (3 DESAFIOS)....

**O que é a fronteira de decisão?**

Pensa que o perceptron precisa responder uma pergunta simples para cada ponto (cada cliente):

“Classifico como 0 (não comprou) ou 1 (comprou)?”

Ele não “enxerga” a cor do ponto. Ele só vê números:

* tempo no site (eixo x)
* valor médio do carrinho (eixo y)


**A fronteira de decisão é a linha no plano onde o neurônio fica “em dúvida”:**

* De um lado da linha, ele responde 0.
* Do outro lado da linha, ele responde 1.

Então, a fronteira de decisão é a fronteira entre duas decisões possíveis:

* “decidir 0” (não compra)
* “decidir 1” (compra)
Tecnicamente: é o conjunto de pontos onde

 * *z = w1x1 + w2x2 = b = 0*

Acima/abaixo dessa linha o sinal de
 muda, e o step muda de 0 para 1.


**Como interpretar no gráfico**

No seu gráfico:

* Pontos vermelhos = clientes que não compraram (classe 0).
* Pontos azuis = clientes que compraram (classe 1).
* Linha verde = fronteira de decisão do perceptron depois do treino.

**A interpretação:**

* Todos os pontos de um lado da linha o modelo marca como 0.
* Todos os pontos do outro lado ele marca como 1.
Ou seja, se você pegasse qualquer coordenada (tempo, valor), alimentasse o perceptron, ele só vai olhar “em que lado da linha” esse ponto cai.

**E quando a linha “não está no lugar certo”?**


No seu gráfico, dá para ver:

* Os vermelhos (0) estão numa nuvem mais embaixo, à esquerda.
* Os azuis (1) estão numa nuvem mais em cima, à direita.
* A linha verde está lá embaixo, inclinada, passando bem longe desses grupos.




**O que isso significa?**

* Os pesos/bias finais não representam bem o problema.
* A linha que eles definem não está separando as nuvens de pontos; está “solta” lá embaixo.

**Muitos pontos serão classificados errado.**
* Se a fronteira de decisão passa longe das duas nuvens, a maior parte dos pontos de uma mesma cor vai cair do mesmo lado da linha — às vezes até todos de uma mesma cor — mas não com a separação desejada:

* Pode acontecer do modelo chamar quase todo mundo de “0” ou quase todo mundo de “1”.
* A acurácia vai ser baixa ou totalmente enviesada para uma classe.
**A posição da linha é um espelho da configuração dos pesos.**
Cada conjunto de pesos (w1,w2,b) gera uma linha diferente:

* se os pesos são ruins → linha ruim → decisões ruins;
*se os peso s aprenderem bem → a linha se mexe e vai se posicionando entre vermelhos e azuis.

**Então, a linha não estar separando bem quer dizer:**

“O treinamento do nosso perceptron, do jeito que foi feito (dados, taxa de aprendizado, épocas, inicialização), não conseguiu encontrar pesos que deixem a linha numa boa posição de separação.”



**Algumas frases importantes (para o entendimento):**

* Imaginem que essa linha verde é a ‘regra’ do modelo: de um lado, ele diz ‘não compra’, do outro lado ele diz ‘compra’. A fronteira de decisão é onde essa regra muda de ideia.

* Se a linha estivesse bem entre os vermelhos e os azuis, teríamos uma boa classificação. Como ela está lá embaixo, longe dos dados, o modelo está tomando decisões ruins.


* Treinar o perceptron é, na prática, mexer nessa linha (girando, subindo, descendo) até ela ficar num lugar que separe bem os grupos.

**Possíveis causas para essa linha ruim**


Sem entrar em muita matemática:

* Poucas épocas: o perceptron não teve “tempo” de ajustar bem os pesos.

* **Taxa de aprendizado inadequada:**

* Muito pequena → quase não mexe a linha.

* Muito grande → pode ficar “pulando” e não converge.

* Inicialização ruim de pesos: às vezes cai numa configuração que demora para sair.
* Implementação da regra: bug no código da parte de atualização.
Isso vira um ótimo gancho para:

* aumentar n_epochs
* mudar learning_rate
* imprimir a acurácia
* E ver a linha se reposicionando até ficar logicamente num lugar mais “entre” os grupos.



#######################################################################

## Desafio 1 – Taxa de aprendizado

1. Altere o valor de `learning_rate` (por exemplo, `0.1`, `0.001`) e rode o treinamento novamente.
2. Observe:
   - A linha de decisão final.
   - O gráfico de erros por época.
3. Pergunta: **Taxas de aprendizado muito altas ou muito baixas atrapalham a convergência? Como?**

## Desafio 2 – Número de épocas

1. Mude `n_epochs` para valores menores (por ex., 5) e maiores (por ex., 100).
2. Veja se o número de erros zera ou se estabiliza.
3. Pergunta: **Vale a pena treinar "para sempre"? O que você observa no gráfico de erros?**

## Desafio 3 – Tente “estragar” a separação

1. Modifique a geração dos dados (célula onde criamos `tempo_0`, `valor_0`, `tempo_1`, `valor_1`).
   - Aproximando os grupos.
   - Misturando mais as distribuições.
2. Pergunta:
   - O perceptron ainda consegue separar bem?
   - Em que situação ele claramente **falha**?

> Dica: isso motiva a ideia de **redes com múltiplas camadas (MLP)**, que você verá nas próximas aulas.

13. Fechamento

## Conclusões da aula

Hoje você:

- Viu como modelar um **neurônio artificial (Perceptron)**:
  - Entradas, pesos, bias, função de ativação.
- Implementou o **feedforward** do perceptron.
- Treinou o modelo ajustando pesos com base no erro (regra de atualização).
- Visualizou a **fronteira de decisão** antes e depois do treino.
- Explorou os efeitos de:
  - Taxa de aprendizado (`learning_rate`).
  - Número de épocas (`n_epochs`).

### Conexão com próximos temas

- O Perceptron é um **bloco básico** das redes neurais.
- Quando empilhamos vários neurônios em várias camadas:
  - Usamos **feedforward** para calcular a saída.
  - Usamos **backpropagation** (uma generalização da ideia de "ajustar pesos" que você viu hoje) para treinar tudo junto.

Na próxima aula, vamos:
- Ver redes com **camadas escondidas (hidden layers)**.
- Falar de forma mais clara sobre **feedforward completo** e **backpropagation**.